# Stage 10 — Observability

**Track A (Buse) · Stage 10 of 10**

| | |
|---|---|
| **Output** | Tracing across the whole system, not only the agent graph |
| **Promotes to** | `src/research_assistant/observability/tracing_B.py` |

> **Ownership note.** Both READMEs put observability in Track A, and you confirmed it
> is yours. The file in the repo is still named `tracing_S.py`, which routes review to
> Sude through `.github/CODEOWNERS`. Rename it to `tracing_B.py` before writing it.

## Why this is stage 10 in the notebooks but early in the build order

The notebooks are ordered by data flow. The build order is not: instrument as soon as
stage 04 works. Tracing is what makes every later bug findable, and retrofitting it
across nine stages costs far more than adding it once.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Backend | Langfuse | LangSmith, plain OpenTelemetry | Self-hostable, and its trace model fits nested retrieval and generation spans without contortion. LangSmith is the smoother option if you are already in that ecosystem. |
| Scope | Every stage from query to draft | Agent graph only | A bad answer is usually a bad chunk. If tracing starts at the agent, the actual cause is invisible and you debug the wrong layer. |
| What each span carries | Query, candidate ids, scores, chosen top-k, latency | Just inputs and outputs | Candidate ids are what let you replay a bad answer against a new ranker without re-running the pipeline. |
| Sampling | Everything | Sample at high volume | Your traffic is one person testing. Sampling would only lose the trace you actually wanted. |

In [ ]:
from _nbsetup_B import REPO, load_cfg
import os, time
from contextlib import contextmanager

# Requires: pip install -e ".[obs]"  and keys in .env (never commit them)
from langfuse import Langfuse
lf = Langfuse(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)
print("langfuse ready")

In [ ]:
# The span shape every retrieval call emits. Keep the field names identical to the
# ones Sude uses in the agent nodes, so one trace reads end to end as a single story.

@contextmanager
def retrieval_span(trace, name, query, **meta):
    span = trace.span(name=name, input={"query": query, **meta})
    t0 = time.perf_counter()
    result = {}
    try:
        yield result
    finally:
        span.end(output={
            "candidate_ids": result.get("candidate_ids", []),
            "top_k_ids": result.get("top_k_ids", []),
            "scores": result.get("scores", []),
            "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
            "ranker": result.get("ranker"),
        })

# Usage inside service_B.search:
#
# with retrieval_span(trace, "hybrid_search", query, fusion="rrf") as out:
#     cands = hybrid_search(query)
#     out["candidate_ids"] = [c["chunk_id"] for c in cands]
#     out["ranker"] = registry.champion

## What to instrument, in order

1. **Ingestion**, once per run: papers in, chunks out, tokens, wall time. Cheap, and it
   dates every index you build.
2. **Each retrieval arm** separately. Fusing two arms into one span hides which arm
   found the answer, which is the question you will ask most often.
3. **The rerank step**, tagged with the active ranker name from the registry. Without
   that tag you cannot tell a baseline trace from a tuned one after the fact.
4. **The MCP tool boundary**, jointly with Sude. Her tools call your service, and this
   is where the two traces have to join into one.
5. **The agent nodes**, hers.

## Exit checks

- [ ] One query produces a single trace spanning retrieval through draft.
- [ ] A trace shows which arm surfaced each cited chunk.
- [ ] The active ranker name appears on every rerank span.
- [ ] No keys in the repo. Langfuse credentials live in `.env`, which is gitignored.
- [ ] `tracing_S.py` has been renamed to `tracing_B.py` and CODEOWNERS routes it to you.